**Türkiye Yapay Zeka Akademisi - Makine Öğrenmesi Final Ödevi**

**Proje: Banka Müşterilerinin Vadeli Mevduat Aboneliği Tahmini**

Amaç:
Bu projenin amacı, banka müşterilerinin bir pazarlama kampanyası
sonucunda vadeli mevduata abone olup olmayacağını tahmin eden
bir sınıflandırma modeli geliştirmektir.

Çalışmada veri inceleme, veri ön işleme, öznitelik mühendisliği,
öznitelik seçimi, model eğitimi, model karşılaştırma,
çapraz doğrulama, hiperparametre optimizasyonu ve model
değerlendirme adımları uygulanacaktır.

Kullanılan kütüphaneler:
- pandas
- numpy
- scikit-learn
- matplotlib
- seaborn

Çalıştırma:
1. Gerekli kütüphanelerin kurulu olduğundan emin olun.
2. bank-full.csv dosyasını Python dosyasıyla aynı klasöre koyun.
3. Python dosyasını çalıştırın.

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [4]:
df = pd.read_csv("bank-full.csv", sep=";")

In [5]:
# Veri setinin ilk 5 gözlemi incelenmektedir.

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [6]:
# Veri setinin satır ve sütun sayısı incelenmektedir.

df.shape

(45211, 17)

In [7]:
# Veri setindeki değişken isimleri incelenmektedir.

df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'balance',
 'housing',
 'loan',
 'contact',
 'day',
 'month',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'y']

In [8]:
# Değişkenlerin veri tipleri incelenmektedir.

df.dtypes

,0
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day,int64


In [9]:
# Sayısal değişkenlerin temel istatistikleri incelenmektedir.

df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [10]:
# Hedef değişkenin sınıf dağılımı incelenmektedir.

df["y"].value_counts()

,count
y,
no,39922
yes,5289


In [11]:
# Hedef değişken sınıflarının yüzdesel dağılımı incelenmektedir.

df["y"].value_counts(normalize=True).mul(100).round(2)

,proportion
y,
no,88.3
yes,11.7


In [12]:
# Meslek değişkenindeki kategoriler incelenmektedir.

df["job"].value_counts()

,count
job,
blue-collar,9732
management,9458
technician,7597
admin.,5171
services,4154
retired,2264
self-employed,1579
entrepreneur,1487
unemployed,1303


In [13]:
# Kategorik değişkenler belirlenmektedir.

categorical_columns = df.select_dtypes(include="object").columns.tolist()

categorical_columns

['job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'poutcome',
 'y']

In [14]:
# Kategorik değişkenlerin kategori sayıları incelenmektedir.

for column in categorical_columns:
    print(f"{column}: {df[column].nunique()} farklı kategori")

job: 12 farklı kategori
marital: 3 farklı kategori
education: 4 farklı kategori
default: 2 farklı kategori
housing: 2 farklı kategori
loan: 2 farklı kategori
contact: 3 farklı kategori
month: 12 farklı kategori
poutcome: 4 farklı kategori
y: 2 farklı kategori


In [15]:
# Kategorik değişkenlerdeki 'unknown' değerlerinin sayısı incelenmektedir.

unknown_counts = {}

for column in categorical_columns:
    unknown_counts[column] = (df[column] == "unknown").sum()

pd.Series(unknown_counts)

,0
job,288
marital,0
education,1857
default,0
housing,0
loan,0
contact,13020
month,0
poutcome,36959
y,0


In [16]:
# Veri setindeki gerçek eksik değerler incelenmektedir.

df.isnull().sum()

,0
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0


In [17]:
# Sayısal değişkenler belirlenmektedir.

numeric_columns = df.select_dtypes(include="number").columns.tolist()

numeric_columns

['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

In [18]:
# Sayısal değişkenlerde IQR yöntemiyle aykırı değerler incelenmektedir.

outlier_counts = {}

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ).sum()

    outlier_counts[column] = outlier_count

pd.Series(outlier_counts).sort_values(ascending=False)

,0
previous,8257
pdays,8257
balance,4729
duration,3235
campaign,3064
age,487
day,0


In [19]:
# Aykırı değer sayısı yüksek olan değişkenlerin temel istatistikleri incelenmektedir.

df[["pdays", "previous", "balance", "duration", "campaign"]].describe()

,pdays,previous,balance,duration,campaign
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.197828,0.580323,1362.272058,258.163080,2.763841
std,100.128746,2.303441,3044.765829,257.527812,3.098021
min,-1.000000,0.000000,-8019.000000,0.000000,1.000000
25%,-1.000000,0.000000,72.000000,103.000000,1.000000
50%,-1.000000,0.000000,448.000000,180.000000,2.000000
75%,-1.000000,0.000000,1428.000000,319.000000,3.000000
max,871.000000,275.000000,102127.000000,4918.000000,63.000000


In [20]:
# Aykırı değerler incelenmiş, ancak gözlemlerin gerçek müşteri davranışlarını temsil edebileceği değerlendirilerek otomatik olarak silinmemiştir.

print("Aykırı değer analizi tamamlanmıştır.")
print("Potansiyel aykırı değerler incelenmiş ve veri bütünlüğünü korumak amacıyla gözlemler korunmuştur.")

Aykırı değer analizi tamamlanmıştır.
Potansiyel aykırı değerler incelenmiş ve veri bütünlüğünü korumak amacıyla gözlemler korunmuştur.


In [21]:
# Görüşme süresinden kategorik bir öznitelik oluşturulmaktadır.

df["duration_group"] = pd.cut(
    df["duration"],
    bins=[-1, 180, 300, np.inf],
    labels=["short", "medium", "long"]
)

In [22]:
# Oluşturulan görüşme süresi gruplarının dağılımı incelenmektedir.

df["duration_group"].value_counts()

,count
duration_group,
short,22660
long,12274
medium,10277


In [23]:
# Müşterinin daha önce kampanyada iletişime geçilip geçilmediğini gösteren yeni bir öznitelik oluşturulmaktadır.

df["previous_contact"] = (df["previous"] > 0).astype(int)

In [24]:
# Daha önce iletişim kurulup kurulmadığına ilişkin dağılım incelenmektedir.

df["previous_contact"].value_counts()

,count
previous_contact,
0,36954
1,8257


In [25]:
# Öznitelik mühendisliği sonrası veri setinin boyutu incelenmektedir.

print("Güncel veri seti boyutu:", df.shape)

print("\nYeni oluşturulan öznitelikler:")
print(df[["duration_group", "previous_contact"]].head())

Güncel veri seti boyutu: (45211, 19)

Yeni oluşturulan öznitelikler:
  duration_group  previous_contact
0         medium                 0
1          short                 0
2          short                 0
3          short                 0
4         medium                 0


In [26]:
# Hedef değişken sayısal forma dönüştürülmektedir.

df["target"] = df["y"].map({
    "no": 0,
    "yes": 1
})

In [27]:
# Sayısallaştırılan hedef değişkenin dağılımı incelenmektedir.

df["target"].value_counts()

,count
target,
0,39922
1,5289


In [28]:
# Sayısal değişkenler ile hedef değişken arasındaki korelasyonlar incelenmektedir.

correlation_with_target = (
    df[numeric_columns + ["target"]]
    .corr()["target"]
    .drop("target")
    .sort_values(key=abs, ascending=False)
)

correlation_with_target

,target
duration,0.394521
pdays,0.103621
previous,0.093236
campaign,-0.073172
balance,0.052838
day,-0.028348
age,0.025155


In [29]:
# Korelasyon analizi sonucunda sayısal değişkenler değerlendirilmiştir.
# Değişkenler arasında çok güçlü bir doğrusal benzerlik bulunmadığından
# sayısal değişkenlerin tamamı modelleme aşamasında korunmaktadır.

selected_numeric_features = numeric_columns.copy()

print("Seçilen sayısal öznitelikler:")
print(selected_numeric_features)

Seçilen sayısal öznitelikler:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


In [30]:
# Modelde kullanılacak özellikler ve hedef değişken ayrılmaktadır.

feature_columns = [
    "age",
    "job",
    "marital",
    "education",
    "default",
    "balance",
    "housing",
    "loan",
    "contact",
    "day",
    "month",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "poutcome",
    "duration_group",
    "previous_contact"
]

X = df[feature_columns]
y = df["target"]

In [31]:
# Modelleme için hazırlanan özellik ve hedef değişken boyutları incelenmektedir.

print("X boyutu:", X.shape)
print("y boyutu:", y.shape)
print("Toplam öznitelik sayısı:", X.shape[1])

X boyutu: (45211, 18)
y boyutu: (45211,)
Toplam öznitelik sayısı: 18


In [32]:
# Veri seti train ve test kümelerine ayrılmaktadır.

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train-validation kümesi tekrar train ve validation olarak ayrılmaktadır.

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=42,
    stratify=y_train_val
)

In [33]:
# Oluşturulan veri kümelerinin boyutları incelenmektedir.

print("Train seti:", X_train.shape)
print("Validation seti:", X_val.shape)
print("Test seti:", X_test.shape)

print("\nTrain hedef dağılımı:")
print(y_train.value_counts())

print("\nValidation hedef dağılımı:")
print(y_val.value_counts())

print("\nTest hedef dağılımı:")
print(y_test.value_counts())

Train seti: (28934, 18)
Validation seti: (7234, 18)
Test seti: (9043, 18)

Train hedef dağılımı:
target
0    25549
1     3385
Name: count, dtype: int64

Validation hedef dağılımı:
target
0    6388
1     846
Name: count, dtype: int64

Test hedef dağılımı:
target
0    7985
1    1058
Name: count, dtype: int64


In [34]:
# Kategorik ve sayısal öznitelikler belirlenmektedir.

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
    "duration_group"
]

numeric_features = [
    "age",
    "balance",
    "day",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "previous_contact"
]

print("Kategorik öznitelikler:")
print(categorical_features)

print("\nSayısal öznitelikler:")
print(numeric_features)

Kategorik öznitelikler:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'duration_group']

Sayısal öznitelikler:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'previous_contact']


In [35]:
# Kategorik ve sayısal değişkenler için veri ön işleme pipeline'ı oluşturulmaktadır.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [36]:
# Ön işleme yapısı yalnızca train verisi üzerinde öğrenilmektedir.

X_train_processed = preprocessor.fit_transform(X_train)

# Öğrenilen dönüşümler validation ve test verilerine uygulanmaktadır.

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train işlenmiş veri boyutu:", X_train_processed.shape)
print("Validation işlenmiş veri boyutu:", X_val_processed.shape)
print("Test işlenmiş veri boyutu:", X_test_processed.shape)

Train işlenmiş veri boyutu: (28934, 55)
Validation işlenmiş veri boyutu: (7234, 55)
Test işlenmiş veri boyutu: (9043, 55)


In [37]:
# Kullanılacak sınıflandırma modelleri tanımlanmaktadır.

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

In [38]:
# Modeller eğitilmekte ve validation verisi üzerinde değerlendirilmektedir.

validation_results = []

trained_models = {}

for model_name, model in models.items():

    # Model eğitilmektedir.
    model.fit(X_train_processed, y_train)

    # Validation tahminleri yapılmaktadır.
    y_val_pred = model.predict(X_val_processed)

    # Model kaydedilmektedir.
    trained_models[model_name] = model

    # Performans metrikleri hesaplanmaktadır.
    validation_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(y_val, y_val_pred, zero_division=0),
        "Recall": recall_score(y_val, y_val_pred, zero_division=0),
        "F1-score": f1_score(y_val, y_val_pred, zero_division=0)
    })

validation_results_df = pd.DataFrame(validation_results)

validation_results_df.sort_values(
    by="F1-score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1-score
2,Random Forest,0.905861,0.657744,0.406619,0.502557
1,KNN,0.897429,0.599237,0.371158,0.458394
0,Logistic Regression,0.903235,0.663677,0.349882,0.458204


In [39]:
# Modellerin 5 katlı stratified cross-validation performansları incelenmektedir.

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for model_name, model in models.items():

    scores = cross_val_score(
        model,
        X_train_processed,
        y_train,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "CV Ortalama F1": scores.mean(),
        "CV Std": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df.sort_values(
    by="CV Ortalama F1",
    ascending=False
)

,Model,CV Ortalama F1,CV Std
2,Random Forest,0.484767,0.019539
0,Logistic Regression,0.454185,0.018676
1,KNN,0.443976,0.025708


In [40]:
# Random Forest için hiperparametre optimizasyonu yapılmaktadır.

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_processed, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='f1', verbose=1)

In [41]:
# Grid Search sonucundaki en iyi hiperparametreler ve CV F1-score incelenmektedir.

print("En iyi hiperparametreler:")
print(grid_search.best_params_)

print("\nEn iyi CV F1-score:")
print(f"{grid_search.best_score_:.4f}")

En iyi hiperparametreler:
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}

En iyi CV F1-score:
0.4918


In [42]:
# Grid Search ile bulunan en iyi model test verisi üzerinde değerlendirilmektedir.

best_model = grid_search.best_estimator_

y_test_pred = best_model.predict(X_test_processed)

print("Test Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nTest Sonuçları")
print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, y_test_pred, zero_division=0):.4f}")

Test Confusion Matrix:
[[7766  219]
 [ 631  427]]

Test Sonuçları
Accuracy : 0.9060
Precision: 0.6610
Recall   : 0.4036
F1-score : 0.5012


In [43]:
# Ön işleme sonrasında oluşan özellik isimleri alınmaktadır.

feature_names = preprocessor.get_feature_names_out()

# Random Forest özellik önemleri hesaplanmaktadır.

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best_model.feature_importances_
})

# En önemli özellikler sıralanmaktadır.

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("En önemli 15 özellik:")
print(feature_importance_df.head(15).to_string(index=False))

En önemli 15 özellik:
                  Feature  Importance
            num__duration    0.206222
                 num__age    0.080339
             num__balance    0.078043
                 num__day    0.069390
    cat__poutcome_success    0.050168
 cat__duration_group_long    0.045587
               num__pdays    0.036130
cat__duration_group_short    0.033629
            num__campaign    0.032442
            num__previous    0.018341
          cat__housing_no    0.015741
           cat__month_mar    0.015514
         cat__housing_yes    0.015041
           cat__month_apr    0.014212
           cat__month_jun    0.013639


In [44]:
# Projenin genel sonuçları özetlenmektedir.

print("\n--- Final Model Değerlendirmesi ---")

print("Seçilen model: Random Forest")
print(f"En iyi CV F1-score: {grid_search.best_score_:.4f}")

print("\nTest performansı:")
print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, y_test_pred, zero_division=0):.4f}")

print("\nEn önemli değişken:")
print(feature_importance_df.iloc[0]["Feature"])


--- Final Model Değerlendirmesi ---
Seçilen model: Random Forest
En iyi CV F1-score: 0.4918

Test performansı:
Accuracy : 0.9060
Precision: 0.6610
Recall   : 0.4036
F1-score : 0.5012

En önemli değişken:
num__duration


**Sonuç ve Değerlendirme**

Validation ve cross-validation sonuçlarında Random Forest diğer modellere göre daha yüksek F1-score elde etmiştir. Bu nedenle hiperparametre optimizasyonu için Random Forest modeli seçilmiştir.

Grid Search sonrasında elde edilen optimize edilmiş model, test setinde:

%90,60 accuracy
%66,10 precision
%40,36 recall
%50,12 F1-score

başarısı göstermiştir.

Modelin accuracy değerinin yüksek olmasına rağmen recall değerinin daha düşük olması, veri setindeki sınıf dengesizliğinin önemli bir etkisi olduğunu göstermektedir. Özellikle müşterinin ürüne abone olduğu yes sınıfını daha iyi yakalamak için gelecekte sınıf ağırlıklandırma, farklı örnekleme yöntemleri veya daha gelişmiş hiperparametre optimizasyonu yöntemleri denenebilir.

Bu proje, veri incelemeden model değerlendirmesine kadar temel bir makine öğrenmesi sürecinin uçtan uca uygulanmasını göstermektedir.